##End-to-end MAG reconstruction

STEP 1. Genome Assembly

In [ ]:
%%bash
wget -O reads.qza \
    https://polybox.ethz.ch/index.php/s/Yw34y55TDX98BA9/download

In [ ]:
%%writefile parallel.config.toml
[parsl]

[[parsl.executors]]
class = "HighThroughputExecutor"
label = "default"

[parsl.executors.provider]
class = "LocalProvider"
max_blocks = 2

In [ ]:
%%bash
qiime assembly assemble-megahit \
    --i-reads reads.qza \
    --p-presets meta-sensitive \
    --p-num-cpu-threads 2 \
    --p-min-contig-len 500 \
    --o-contigs contigs.qza \
    --parallel-config parallel.config.toml \
    --verbose


In [ ]:
%%bash
wget -O reference-genomes.qza \
    https://polybox.ethz.ch/index.php/s/jA9FB8EF4YjP82x/download

In [ ]:
%%bash
qiime assembly evaluate-quast \
    --i-contigs contigs.qza \
    --i-references reference-genomes.qza \
    --p-threads 4 \
    --p-min-contig 500 \
    --o-reference-genomes ref-genomes.qza \
    --o-results-table quast-results.qza \
    --o-visualization contigs-qc-quast.qzv \
    --verbose


STEP 2. MAG Binning
i.Contig indexing
ii.Read mapping
iii. Contig binning
iv. MAG quality control

In [ ]:
%%bash
qiime assembly index-contigs \
    --i-contigs contigs.qza \
    --p-threads 2 \
    --p-seed 100 \
    --o-index contigs-index.qza \
    --parallel-config parallel.config.toml \
    --verbose


In [ ]:
%%bash
qiime assembly map-reads \
    --i-index contigs-index.qza \
    --i-reads reads.qza \
    --p-threads 2 \
    --p-seed 100 \
    --o-alignment-maps reads-to-contigs-aln.qza \
    --parallel-config parallel.config.toml \
    --verbose


In [ ]:
%%bash
qiime annotate bin-contigs-metabat \
    --i-contigs contigs.qza \
    --i-alignment-maps reads-to-contigs-aln.qza \
    --p-num-threads 4 \
    --p-seed 100 \
    --o-mags mags.qza \
    --o-contig-map contig-map.qza \
    --o-unbinned-contigs unbinned-contigs.qza \
    --verbose


In [ ]:
%%bash
qiime annotate fetch-busco-db \
    --p-lineages bacteria_odb12 \
    --o-db busco-db-bacteria.qza \
    --verbose


In [ ]:
%%bash
qiime annotate evaluate-busco \
    --i-mags mags.qza \
    --i-db busco-db-bacteria.qza \
    --i-unbinned-contigs unbinned-contigs.qza \
    --p-lineage-dataset bacteria_odb12 \
    --p-cpu 2 \
    --o-results busco-results.qza \
    --o-visualization mags.qzv \
    --parallel-config parallel.config.toml \
    --verbose


In [ ]:
%%bash
qiime annotate filter-mags \
    --i-mags mags.qza \
    --m-metadata-file busco-results.qza \
    --p-where "completeness>50 AND contamination<10" \
    --p-on "mag" \
    --o-filtered-mags mags-filtered.qza \
    --verbose


STEP 3. MAG Dereplication

In [ ]:
%%bash
qiime sourmash compute \
    --i-sequence-file mags-filtered.qza \
    --p-ksizes 105 \
    --p-scaled 100 \
    --o-min-hash-signature min-hash.qza \
    --verbose

In [ ]:
%%bash
qiime sourmash compare \
    --i-min-hash-signature min-hash.qza \
    --p-ksize 105 \
    --o-compare-output min-hash-compare.qza \
    --verbose

STEP 4. Taxonomic classification
i.Database preparation
ii.Read-based classification
iii. MAG-based classification

In [ ]:
%%bash
qiime annotate dereplicate-mags \
    --i-mags mags-filtered.qza \
    --i-distance-matrix min-hash-compare.qza \
    --m-metadata-file busco-results.qza \
    --p-metadata-column completeness \
    --p-threshold 0.9 \
    --o-dereplicated-mags mags-derep.qza \
    --o-table table.qza \
    --verbose

In [ ]:
%%bash
qiime annotate build-kraken-db \
    --p-collection standard8 \
    --o-kraken2-db kraken2-db.qza \
    --o-bracken-db bracken-db.qza \
    --verbose


In [ ]:
%%bash
qiime annotate classify-kraken2 \
    --i-seqs reads.qza \
    --i-db kraken2-db.qza \
    --p-threads 4 \
    --p-memory-mapping \
    --o-reports kraken2-reports-reads.qza \
    --o-outputs kraken2-hits-reads.qza \
    --parallel-config parallel.config.toml \
    --verbose


In [ ]:
%%bash
qiime annotate estimate-bracken \
    --i-kraken2-reports kraken2-reports-reads.qza \
    --i-db bracken-db.qza \
    --p-read-len 150 \
    --o-reports bracken-reports.qza \
    --o-taxonomy bracken-taxonomy.qza \
    --o-table bracken-table.qza \
    --verbose


In [ ]:
%%bash
qiime taxa barplot \
    --i-table bracken-table.qza \
    --i-taxonomy bracken-taxonomy.qza \
    --o-visualization bracken-barplot.qzv \
    --verbose


In [ ]:
%%bash
qiime annotate classify-kraken2 \
    --i-seqs mags-derep.qza \
    --i-db kraken2-db.qza \
    --p-threads 4 \
    --p-memory-mapping \
    --o-reports kraken2-reports-mags.qza \
    --o-outputs kraken2-hits-mags.qza \
    --parallel-config parallel.config.toml \
    --verbose


In [ ]:
%%bash
qiime annotate kraken2-to-mag-features \
    --i-reports kraken2-reports-mags.qza \
    --i-outputs kraken2-hits-mags.qza \
    --o-taxonomy mags-taxonomy.qza \
    --verbose


STEP 5. MAG abundance estimation
i.MAG indexing
ii.Read mapping
iii. MAG length estimation
iv.Abundance estimation
v.Taxonomic composition visualization

In [ ]:
%%bash
qiime assembly index-derep-mags \
    --i-mags mags-derep.qza \
    --p-threads 4 \
    --p-seed 100 \
    --o-index mags-derep-index.qza \
    --verbose


In [ ]:
%%bash
qiime assembly map-reads \
    --i-index mags-derep-index.qza \
    --i-reads reads.qza \
    --p-threads 2 \
    --p-seed 100 \
    --o-alignment-maps reads-to-mags-aln.qza \
    --parallel-config parallel.config.toml \
    --verbose


In [ ]:
%%bash
qiime annotate get-feature-lengths \
    --i-features mags-derep.qza \
    --o-lengths mags-derep-lengths.qza \
    --verbose


In [ ]:
%%bash
qiime annotate estimate-abundance \
    --i-alignment-maps reads-to-mags-aln.qza \
    --i-feature-lengths mags-derep-lengths.qza \
    --p-metric tpm \
    --p-min-mapq 42 \
    --p-threads 4 \
    --o-abundances mags-abundances.qza \
    --verbose


In [ ]:
%%bash
qiime taxa barplot \
    --i-table mags-abundances.qza \
    --i-taxonomy mags-taxonomy.qza \
    --o-visualization mags-taxa-barplot.qzv \
    --verbose
